# UTS Data Acquisition & Management
## Akuisisi dan Integrasi Data Pariwisata Indonesia

### Identitas
- Nama: Putu Reva Dhamma Nanda
- NIM: 2501010167
- Mata Kuliah: Data Acquisition & Management
- Kelas: F

### 1. Pendahuluan
Proyek ini membahas proses akuisisi dan pengelolaan data destinasi
pariwisata di Indonesia. Data dikumpulkan dari beberapa sumber dengan
menggunakan metode pengunduhan dataset terbuka, API, dan web scraping.

Dataset dari berbagai sumber akan diperiksa karakteristiknya, disimpan
sebagai data mentah, serta diproses untuk keperluan integrasi data.

### 2. Tujuan
1. Mengumpulkan data pariwisata dari berbagai sumber.
2. Mengidentifikasi karakteristik dan struktur dataset.
3. Memeriksa lisensi serta etika penggunaan data.
4. Melakukan akuisisi data menggunakan API dan web scraping.
5. Menggabungkan data dari beberapa sumber menggunakan metode merge.

### 3. Metode Akuisisi
1. Dataset terbuka: Indonesia Tourism Destination.
2. API: Data cuaca menggunakan API publik.
3. Web scraping: Mengambil data dari halaman web statis yang diizinkan.

### 4. Struktur Proyek
Proyek ini menggunakan struktur folder sebagai berikut:
1. data/raw: menyimpan data mentah yang diperoleh dari dataset utama, API, dan web scraping.
2. data/processed: menyimpan data yang telah dibersihkan, dinormalisasi, dan dataset hasil integrasi.
3. docs: menyimpan dokumentasi sumber data (provenance), checklist etika data, pemeriksaan robots.txt, dan metadata integrasi.
Dataset final hasil integrasi disimpan dalam file data/processed/tourism_integrated.csv.

In [3]:
# Import library yang dibutuhkan
import os
import pandas as pd
import numpy as np
import requests
import hashlib
import json
import time

# Membuat struktur folder proyek
folders = [
    "data/raw",
    "data/processed",
    "docs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Struktur folder proyek berhasil dibuat.")
print("Folder yang tersedia:", folders)

Struktur folder proyek berhasil dibuat.
Folder yang tersedia: ['data/raw', 'data/processed', 'docs']


In [4]:
# Upload dataset utama
from google.colab import files
import shutil
import os

uploaded = files.upload()

file_name = "tourism_with_id.csv"
dataset_path = "data/raw/tourism_with_id.csv"

if os.path.exists(file_name):
    shutil.move(file_name, dataset_path)
    print("Dataset berhasil disimpan:", dataset_path)
else:
    print("File dataset tidak ditemukan.")

Saving tourism_with_id.csv to tourism_with_id.csv
Dataset berhasil disimpan: data/raw/tourism_with_id.csv


In [5]:
# Membaca dataset utama dari folder data/raw
dataset_path = "data/raw/tourism_with_id.csv"

df = pd.read_csv(dataset_path)

df.head()

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156,NaN,4
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134,NaN,5


In [6]:
jumlah_baris, jumlah_kolom = df.shape

print("Jumlah baris:", jumlah_baris)
print("Jumlah kolom:", jumlah_kolom)
print("Jumlah data:", df.shape)

Jumlah baris: 437
Jumlah kolom: 13
Jumlah data: (437, 13)


In [7]:
# Menampilkan nama kolom dataset
print("Nama kolom:")
print(df.columns.tolist())

print("\nTipe data setiap kolom:")
print(df.dtypes)

Nama kolom:
['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Unnamed: 11', 'Unnamed: 12']

Tipe data setiap kolom:
Place_Id          int64
Place_Name       object
Description      object
Category         object
City             object
Price             int64
Rating          float64
Time_Minutes    float64
Coordinate       object
Lat             float64
Long            float64
Unnamed: 11     float64
Unnamed: 12       int64
dtype: object


In [8]:
# Memeriksa jumlah data kosong pada setiap kolom
missing_data = df.isnull().sum()

print("Jumlah missing value setiap kolom:")
print(missing_data)

print("\nTotal missing value:", missing_data.sum())

Jumlah missing value setiap kolom:
Place_Id          0
Place_Name        0
Description       0
Category          0
City              0
Price             0
Rating            0
Time_Minutes    232
Coordinate        0
Lat               0
Long              0
Unnamed: 11     437
Unnamed: 12       0
dtype: int64

Total missing value: 669


In [9]:
# Menghitung ukuran file dataset
file_size_bytes = os.path.getsize(dataset_path)

file_size_kb = file_size_bytes / 1024
file_size_mb = file_size_bytes / (1024 * 1024)

print("Ukuran file dalam Bytes:", file_size_bytes)
print("Ukuran file dalam KB:", round(file_size_kb, 2))
print("Ukuran file dalam MB:", round(file_size_mb, 4))

Ukuran file dalam Bytes: 365796
Ukuran file dalam KB: 357.22
Ukuran file dalam MB: 0.3489


In [10]:
# Estimasi kebutuhan storage selama 1 tahun
# Asumsi: ukuran dataset setiap bulan tetap sama
storage_12_bulan = file_size_mb * 12
# Menambahkan cadangan storage sebesar 30%
storage_dengan_cadangan = storage_12_bulan * 1.30

print("Ukuran dataset awal:", round(file_size_mb, 4), "MB")
print("Estimasi storage 12 bulan:", round(storage_12_bulan, 4), "MB")
print("Estimasi storage + cadangan 30%:",
      round(storage_dengan_cadangan, 4), "MB")

Ukuran dataset awal: 0.3489 MB
Estimasi storage 12 bulan: 4.1862 MB
Estimasi storage + cadangan 30%: 5.4421 MB


In [11]:
# Pemeriksaan kolom tambahan pada dataset
kolom_tambahan = ["Unnamed: 11", "Unnamed: 12"]

for kolom in kolom_tambahan:
    print(f"\n=== {kolom} ===")

    print("Contoh data:")
    display(df[[kolom]].head(10))

    print("\nJumlah nilai:")
    display(df[kolom].value_counts(dropna=False))


=== Unnamed: 11 ===
Contoh data:


,Unnamed: 11
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN



Jumlah nilai:


,count
Unnamed: 11,
NaN,437



=== Unnamed: 12 ===
Contoh data:


,Unnamed: 12
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10



Jumlah nilai:


,count
Unnamed: 12,
1,1
2,1
3,1
4,1
5,1
...,...
433,1
434,1
435,1


In [12]:
# Analisis awal dataset pariwisata
print("=== CAKUPAN KOTA ===")
print("Daftar kota:", df["City"].unique())

print("\nJumlah destinasi per kota:")
display(df["City"].value_counts())

print("\n=== KATEGORI DESTINASI ===")
print("Daftar kategori:", df["Category"].unique())

print("\nJumlah destinasi per kategori:")
display(df["Category"].value_counts())

print("\n=== STATISTIK HARGA TIKET ===")
display(df["Price"].describe())

print("\n=== STATISTIK RATING ===")
display(df["Rating"].describe())

=== CAKUPAN KOTA ===
Daftar kota: ['Jakarta' 'Yogyakarta' 'Bandung' 'Semarang' 'Surabaya']

Jumlah destinasi per kota:


,count
City,
Yogyakarta,126
Bandung,124
Jakarta,84
Semarang,57
Surabaya,46



=== KATEGORI DESTINASI ===
Daftar kategori: ['Budaya' 'Taman Hiburan' 'Cagar Alam' 'Bahari' 'Pusat Perbelanjaan'
 'Tempat Ibadah']

Jumlah destinasi per kategori:


,count
Category,
Taman Hiburan,135
Budaya,117
Cagar Alam,106
Bahari,47
Tempat Ibadah,17
Pusat Perbelanjaan,15



=== STATISTIK HARGA TIKET ===


,Price
count,437.000000
mean,24652.173913
std,66446.374709
min,0.000000
25%,0.000000
50%,5000.000000
75%,20000.000000
max,900000.000000



=== STATISTIK RATING ===


,Rating
count,437.000000
mean,4.442792
std,0.208587
min,3.400000
25%,4.300000
50%,4.500000
75%,4.600000
max,5.000000


In [13]:
# Membuat provenance untuk dataset utama
import os
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo

dataset_path = "data/raw/tourism_with_id.csv"

os.makedirs("docs", exist_ok=True)

file_size = os.path.getsize(dataset_path)

md5_hash = hashlib.md5()

with open(dataset_path, "rb") as file:
    for chunk in iter(lambda: file.read(4096), b""):
        md5_hash.update(chunk)

file_md5 = md5_hash.hexdigest()

# Waktu akuisisi menggunakan zona waktu Indonesia
waktu_akses = datetime.now(ZoneInfo("Asia/Makassar")).isoformat()

# Identitas pengambil data
nama_pengambil = input("Masukkan nama lengkap: ")
nim_pengambil = input("Masukkan NIM: ")

# Metadata provenance
provenance = {
    "file_name": os.path.basename(dataset_path),
    "file_size_bytes": file_size,
    "file_size_kb": round(file_size / 1024, 2),
    "md5": file_md5,
    "source_name": "Indonesia Tourism Destination",
    "publisher": "A_Prabowo (Kaggle)",
    "source_url": "https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination",
    "acquisition_method": "Download dataset CSV",
    "access_time": waktu_akses,
    "license": "Belum terverifikasi; periksa informasi lisensi pada halaman Kaggle",
    "collector_name": nama_pengambil,
    "collector_nim": nim_pengambil
}

# Menyimpan provenance ke file JSON
with open("docs/provenance.json", "w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=4, ensure_ascii=False)

print("Provenance berhasil dibuat!")
print(json.dumps(provenance, indent=4, ensure_ascii=False))

Masukkan nama lengkap: Putu Reva Dhamma Nanda
Masukkan NIM: 2501010167
Provenance berhasil dibuat!
{
    "file_name": "tourism_with_id.csv",
    "file_size_bytes": 365796,
    "file_size_kb": 357.22,
    "md5": "722ca7f02aa2f847c41798c3f6a5d7a5",
    "source_name": "Indonesia Tourism Destination",
    "publisher": "A_Prabowo (Kaggle)",
    "source_url": "https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination",
    "acquisition_method": "Download dataset CSV",
    "access_time": "2026-11-26T19:09:01.309405+08:00",
    "license": "Belum terverifikasi; periksa informasi lisensi pada halaman Kaggle",
    "collector_name": "Putu Reva Dhamma Nanda",
    "collector_nim": "2501010167"
}


In [14]:
# Data Ethics Checklist dan pemeriksaan PII
import pandas as pd
import os

# Membuat Data Ethics Checklist
ethics_checklist = pd.DataFrame([
    {
        "No": 1,
        "Aspek": "Legalitas sumber",
        "Pertanyaan": "Apakah dataset berasal dari sumber yang dapat diakses dan digunakan secara sah?",
        "Penilaian": "Perlu verifikasi",
        "Bukti_Catatan": "Dataset diunduh dari Kaggle; ketentuan penggunaan perlu diperiksa."
    },
    {
        "No": 2,
        "Aspek": "Lisensi",
        "Pertanyaan": "Apakah lisensi dataset sudah diperiksa dan dicatat?",
        "Penilaian": "Perlu verifikasi",
        "Bukti_Catatan": "Lisensi belum dipastikan; perlu diperiksa pada halaman Kaggle."
    },
    {
        "No": 3,
        "Aspek": "Consent",
        "Pertanyaan": "Apakah pengumpulan data memerlukan persetujuan subjek data?",
        "Penilaian": "Tidak relevan untuk data destinasi publik",
        "Bukti_Catatan": "Dataset berisi informasi destinasi wisata, bukan data responden."
    },
    {
        "No": 4,
        "Aspek": "Minimalisasi data",
        "Pertanyaan": "Apakah data yang dikumpulkan hanya sesuai kebutuhan tujuan proyek?",
        "Penilaian": "Terpenuhi",
        "Bukti_Catatan": "Data digunakan untuk identifikasi destinasi dan integrasi data pariwisata."
    },
    {
        "No": 5,
        "Aspek": "PII dan privasi",
        "Pertanyaan": "Apakah terdapat PII seperti nama pribadi, email, nomor telepon, atau identitas?",
        "Penilaian": "Perlu pemeriksaan kolom",
        "Bukti_Catatan": "Kolom dataset akan diperiksa untuk indikasi data pribadi."
    },
    {
        "No": 6,
        "Aspek": "Transparansi tujuan",
        "Pertanyaan": "Apakah tujuan pengumpulan dan penggunaan data dijelaskan?",
        "Penilaian": "Terpenuhi",
        "Bukti_Catatan": "Tujuan proyek dijelaskan pada notebook."
    },
    {
        "No": 7,
        "Aspek": "Atribusi sumber",
        "Pertanyaan": "Apakah sumber data dan penerbit dicatat untuk atribusi?",
        "Penilaian": "Terpenuhi",
        "Bukti_Catatan": "Nama sumber dan URL dicatat pada provenance."
    },
    {
        "No": 8,
        "Aspek": "Kepatuhan ketentuan sumber",
        "Pertanyaan": "Apakah ketentuan portal dan batas penggunaan sumber diperiksa?",
        "Penilaian": "Perlu verifikasi",
        "Bukti_Catatan": "Ketentuan penggunaan dataset Kaggle perlu diperiksa."
    },
    {
        "No": 9,
        "Aspek": "Risiko bias dan kualitas",
        "Pertanyaan": "Apakah keterbatasan cakupan dan potensi bias dataset dicatat?",
        "Penilaian": "Terpenuhi",
        "Bukti_Catatan": "Dataset mencakup lima kota dan enam kategori destinasi."
    }
])

# Pemeriksaan indikasi PII dari nama kolom
kolom_pii_personal = [
    kolom for kolom in df.columns
    if any(kata in kolom.lower() for kata in [
        "email", "phone", "address", "contact",
        "person", "identity", "nik"
    ])
]

# Memperbarui hasil pemeriksaan PII
if len(kolom_pii_personal) == 0:
    penilaian_pii = "Tidak ditemukan indikasi PII"
    bukti_pii = (
        "Pemeriksaan nama kolom tidak menemukan kolom email, "
        "telepon, alamat pribadi, atau identitas individu. "
        "Place_Name berisi nama destinasi wisata, bukan nama orang. "
        "Pemeriksaan dilakukan berdasarkan nama kolom dan sampel data."
    )
else:
    penilaian_pii = "Perlu pemeriksaan lanjutan"
    bukti_pii = (
        "Kolom yang perlu diperiksa lebih lanjut: "
        + ", ".join(kolom_pii_personal)
    )

ethics_checklist.loc[
    ethics_checklist["No"] == 5, "Penilaian"
] = penilaian_pii

ethics_checklist.loc[
    ethics_checklist["No"] == 5, "Bukti_Catatan"
] = bukti_pii

# Menampilkan checklist
display(ethics_checklist)

# Menyimpan checklist
os.makedirs("docs", exist_ok=True)

ethics_checklist.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Data Ethics Checklist berhasil diperiksa dan disimpan.")
print("Lokasi: docs/data_ethics_checklist.csv")

,No,Aspek,Pertanyaan,Penilaian,Bukti_Catatan
0,1,Legalitas sumber,Apakah dataset berasal dari sumber yang dapat ...,Perlu verifikasi,Dataset diunduh dari Kaggle; ketentuan penggun...
1,2,Lisensi,Apakah lisensi dataset sudah diperiksa dan dic...,Perlu verifikasi,Lisensi belum dipastikan; perlu diperiksa pada...
2,3,Consent,Apakah pengumpulan data memerlukan persetujuan...,Tidak relevan untuk data destinasi publik,"Dataset berisi informasi destinasi wisata, buk..."
3,4,Minimalisasi data,Apakah data yang dikumpulkan hanya sesuai kebu...,Terpenuhi,Data digunakan untuk identifikasi destinasi da...
4,5,PII dan privasi,"Apakah terdapat PII seperti nama pribadi, emai...",Tidak ditemukan indikasi PII,Pemeriksaan nama kolom tidak menemukan kolom e...
5,6,Transparansi tujuan,Apakah tujuan pengumpulan dan penggunaan data ...,Terpenuhi,Tujuan proyek dijelaskan pada notebook.
6,7,Atribusi sumber,Apakah sumber data dan penerbit dicatat untuk ...,Terpenuhi,Nama sumber dan URL dicatat pada provenance.
7,8,Kepatuhan ketentuan sumber,Apakah ketentuan portal dan batas penggunaan s...,Perlu verifikasi,Ketentuan penggunaan dataset Kaggle perlu dipe...
8,9,Risiko bias dan kualitas,Apakah keterbatasan cakupan dan potensi bias d...,Terpenuhi,Dataset mencakup lima kota dan enam kategori d...


Data Ethics Checklist berhasil diperiksa dan disimpan.
Lokasi: docs/data_ethics_checklist.csv


In [15]:
# Akuisisi data cuaca API Open-Meteo untuk 30 destinasi
import requests
import pandas as pd
import json
import os
import time
from datetime import datetime
from zoneinfo import ZoneInfo

# Input email untuk User-Agent
email = input("Masukkan email: ")

# Memilih 30 destinasi dengan koordinat valid
df_api = df.dropna(subset=["Lat", "Long"]).head(30).copy()

print("Jumlah destinasi yang akan diambil:", len(df_api))

url_api = "https://api.open-meteo.com/v1/forecast"

headers = {
    "User-Agent": f"UTS-DAM/1.0 ({email})"
}

hasil_api = []
raw_api = []

# Request data cuaca
for index, row in df_api.iterrows():

    latitude = float(row["Lat"])
    longitude = float(row["Long"])

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "precipitation,"
            "wind_speed_10m"
        ),
        "timezone": "Asia/Jakarta"
    }

    try:
        response = requests.get(
            url_api,
            params=params,
            headers=headers,
            timeout=(10, 30)
        )

        response.raise_for_status()
        data = response.json()

        # Menyimpan respons API mentah
        raw_api.append({
            "place_id": row["Place_Id"],
            "place_name": row["Place_Name"],
            "request_url": response.url,
            "access_time": datetime.now(
                ZoneInfo("Asia/Makassar")
            ).isoformat(),
            "response_json": data
        })

        current = data.get("current", {})

        # Menormalisasi data API
        hasil_api.append({
            "Place_Id": row["Place_Id"],
            "Place_Name": row["Place_Name"],
            "City": row["City"],
            "Category": row["Category"],
            "Latitude": latitude,
            "Longitude": longitude,
            "Temperature_C": current.get("temperature_2m"),
            "Humidity_Percent": current.get("relative_humidity_2m"),
            "Precipitation_mm": current.get("precipitation"),
            "Wind_Speed_kmh": current.get("wind_speed_10m"),
            "Weather_Time": current.get("time"),
            "Timezone": data.get("timezone"),
            "API_Status": response.status_code
        })

        print(f"Berhasil: {row['Place_Name']}")

    except Exception as error:
        print(f"Gagal: {row['Place_Name']} - {error}")

    time.sleep(1)

# Membuat DataFrame
df_weather = pd.DataFrame(hasil_api)

# Menyimpan raw JSON
os.makedirs("data/raw", exist_ok=True)

with open(
    "data/raw/open_meteo_raw.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(raw_api, file, indent=4, ensure_ascii=False)

# Menyimpan data API yang telah dinormalisasi
os.makedirs("data/processed", exist_ok=True)

normalized_api_path = "data/processed/open_meteo_normalized.csv"

df_weather.to_csv(
    normalized_api_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n================================")
print("AKUISISI API SELESAI")
print("Jumlah data berhasil:", len(df_weather))
print("================================")

display(df_weather.head())

Masukkan email: puturevadhammananda@gmail.com
Jumlah destinasi yang akan diambil: 30
Gagal: Monumen Nasional - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Kota Tua - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Dunia Fantasi - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Taman Mini Indonesia Indah (TMII) - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Atlantis Water Adventure - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Taman Impian Jaya Ancol - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Kebun Binatang Ragunan - HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Gagal: Ocean Ecopark - HTTPSConnectionPool(host='api.open-mete

,Place_Id,Place_Name,City,Category,Latitude,Longitude,Temperature_C,Humidity_Percent,Precipitation_mm,Wind_Speed_kmh,Weather_Time,Timezone,API_Status
0,11,Pulau Bidadari,Jakarta,Bahari,-6.035833,106.746944,27.4,81,0.1,10.1,2026-11-26T18:00,Asia/Jakarta,200
1,12,Pulau Pari,Jakarta,Bahari,-5.907433,106.586399,28.1,84,0.1,12.3,2026-11-26T18:00,Asia/Jakarta,200
2,13,Pulau Pramuka,Jakarta,Bahari,-5.745962,106.613658,28.3,80,0.1,10.6,2026-11-26T18:00,Asia/Jakarta,200
3,14,Pulau Pelangi,Jakarta,Bahari,-5.587055,106.588500,28.9,77,0.1,15.9,2026-11-26T18:00,Asia/Jakarta,200
4,15,Pasar Seni,Jakarta,Pusat Perbelanjaan,-6.176687,106.841767,26.3,90,0.1,7.4,2026-11-26T18:00,Asia/Jakarta,200


In [16]:
# Validasi hasil akuisisi API Open-Meteo
import os

# Jumlah data dan destinasi
print("Jumlah baris tabel API:", len(df_weather))
print("Jumlah Place_Id unik:", df_weather["Place_Id"].nunique())

# Duplikasi
print("\nJumlah duplikasi baris:", df_weather.duplicated().sum())
print("Jumlah Place_Id duplikat:",
      df_weather["Place_Id"].duplicated().sum())

# Missing value
print("\nMissing value:")
display(df_weather.isnull().sum())

# Status API
print("\nStatus API:")
display(df_weather["API_Status"].value_counts())

# Pemeriksaan file
file_raw_json = "data/raw/open_meteo_raw.json"
file_normalized_csv = "data/processed/open_meteo_normalized.csv"

print("\nPemeriksaan file:")
print("Raw JSON tersedia:", os.path.exists(file_raw_json))
print("Normalized CSV tersedia:", os.path.exists(file_normalized_csv))

# Ukuran file
if os.path.exists(file_raw_json):
    print("Ukuran Raw JSON (KB):",
          round(os.path.getsize(file_raw_json) / 1024, 2))

if os.path.exists(file_normalized_csv):
    print("Ukuran Normalized CSV (KB):",
          round(os.path.getsize(file_normalized_csv) / 1024, 2))

Jumlah baris tabel API: 20
Jumlah Place_Id unik: 20

Jumlah duplikasi baris: 0
Jumlah Place_Id duplikat: 0

Missing value:


,0
Place_Id,0
Place_Name,0
City,0
Category,0
Latitude,0
Longitude,0
Temperature_C,0
Humidity_Percent,0
Precipitation_mm,0
Wind_Speed_kmh,0



Status API:


,count
API_Status,
200,20



Pemeriksaan file:
Raw JSON tersedia: True
Normalized CSV tersedia: True
Ukuran Raw JSON (KB): 23.99
Ukuran Normalized CSV (KB): 2.34


In [17]:
# Menyimpan hasil API dalam format JSON tabular

json_table_path = "data/processed/weather_table.json"

df_weather.to_json(
    json_table_path,
    orient="records",
    indent=4,
    force_ascii=False
)

print("JSON tabular berhasil disimpan:", json_table_path)

# Menampilkan contoh hasil JSON
with open(json_table_path, "r", encoding="utf-8") as file:
    json_table = json.load(file)

print("\nContoh isi JSON:")
print(json.dumps(json_table[:2], indent=4, ensure_ascii=False))

JSON tabular berhasil disimpan: data/processed/weather_table.json

Contoh isi JSON:
[
    {
        "Place_Id": 11,
        "Place_Name": "Pulau Bidadari",
        "City": "Jakarta",
        "Category": "Bahari",
        "Latitude": -6.035833,
        "Longitude": 106.746944,
        "Temperature_C": 27.4,
        "Humidity_Percent": 81,
        "Precipitation_mm": 0.1,
        "Wind_Speed_kmh": 10.1,
        "Weather_Time": "2026-11-26T18:00",
        "Timezone": "Asia/Jakarta",
        "API_Status": 200
    },
    {
        "Place_Id": 12,
        "Place_Name": "Pulau Pari",
        "City": "Jakarta",
        "Category": "Bahari",
        "Latitude": -5.9074328,
        "Longitude": 106.5863989,
        "Temperature_C": 28.1,
        "Humidity_Percent": 84,
        "Precipitation_mm": 0.1,
        "Wind_Speed_kmh": 12.3,
        "Weather_Time": "2026-11-26T18:00",
        "Timezone": "Asia/Jakarta",
        "API_Status": 200
    }
]


In [18]:
# Provenance sumber API Open-Meteo
import os
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo

raw_api_path = "data/raw/open_meteo_raw.json"
normalized_api_path = "data/processed/open_meteo_normalized.csv"

# Fungsi menghitung MD5
def hitung_md5(path):
    md5 = hashlib.md5()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            md5.update(chunk)

    return md5.hexdigest()

# Metadata API
provenance_api = {
    "source_name": "Open-Meteo Weather API",
    "publisher": "Open-Meteo",
    "source_url": "https://api.open-meteo.com/v1/forecast",
    "documentation_url": "https://open-meteo.com/en/docs",
    "license": "CC BY 4.0",
    "license_url": "https://creativecommons.org/licenses/by/4.0/",
    "terms_url": "https://open-meteo.com/en/terms",
    "acquisition_method": "HTTP GET Request menggunakan Python Requests",
    "request_type": "Current Weather Forecast",
    "number_of_locations": len(df_weather),
    "variables": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    ],
    "access_time": datetime.now(
        ZoneInfo("Asia/Makassar")
    ).isoformat(),
    "collector_name": nama_pengambil,
    "collector_nim": nim_pengambil,
    "user_agent": f"UTS-DAM/1.0 ({email})",
    "raw_file": {
        "file_name": os.path.basename(raw_api_path),
        "file_size_bytes": os.path.getsize(raw_api_path),
        "md5": hitung_md5(raw_api_path)
    },
    "normalized_file": {
        "file_name": os.path.basename(normalized_api_path),
        "file_size_bytes": os.path.getsize(normalized_api_path),
        "md5": hitung_md5(normalized_api_path)
    },
    "processing_notes": (
        "Raw JSON disimpan tanpa perubahan. Data cuaca dinormalisasi "
        "menjadi tabel CSV dan dilengkapi identitas destinasi dari "
        "dataset utama untuk integrasi selanjutnya."
    ),
    "attribution": "Weather data by Open-Meteo.com"
}

# Menyimpan provenance
os.makedirs("docs", exist_ok=True)

with open(
    "docs/provenance_api.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        provenance_api,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Provenance API berhasil dibuat!")
print(json.dumps(provenance_api, indent=4, ensure_ascii=False))

Provenance API berhasil dibuat!
{
    "source_name": "Open-Meteo Weather API",
    "publisher": "Open-Meteo",
    "source_url": "https://api.open-meteo.com/v1/forecast",
    "documentation_url": "https://open-meteo.com/en/docs",
    "license": "CC BY 4.0",
    "license_url": "https://creativecommons.org/licenses/by/4.0/",
    "terms_url": "https://open-meteo.com/en/terms",
    "acquisition_method": "HTTP GET Request menggunakan Python Requests",
    "request_type": "Current Weather Forecast",
    "number_of_locations": 20,
    "variables": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m"
    ],
    "access_time": "2026-11-26T19:15:14.761848+08:00",
    "collector_name": "Putu Reva Dhamma Nanda",
    "collector_nim": "2501010167",
    "user_agent": "UTS-DAM/1.0 (puturevadhammananda@gmail.com)",
    "raw_file": {
        "file_name": "open_meteo_raw.json",
        "file_size_bytes": 24569,
        "md5": "bba57bb62eb675fb6f3ec9

In [19]:
# Pemeriksaan robots.txt sebelum scraping halaman target
import requests
import time
import json
import os
from urllib.robotparser import RobotFileParser
from datetime import datetime
from zoneinfo import ZoneInfo

# Memastikan variabel email dan url_scraping telah didefinisikan
try:
    email
except NameError:
    email = input("Masukkan email Anda untuk User-Agent: ")

try:
    url_scraping
except NameError:
    url_scraping = "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population"

# URL robots.txt dan halaman target
robots_url = "https://en.wikipedia.org/robots.txt"

# User-Agent yang sama dengan request scraping
headers_scraping = {
    "User-Agent": f"UTS-DAM/1.0 ({email})"
}

# Mengambil robots.txt
response_robots = requests.get(
    robots_url,
    headers=headers_scraping,
    timeout=30
)

print("Status robots.txt:", response_robots.status_code)

if response_robots.status_code == 200:

    # Membaca aturan robots.txt
    robot_parser = RobotFileParser()
    robot_parser.set_url(robots_url)
    robot_parser.parse(response_robots.text.splitlines())

    # Memeriksa apakah halaman target diizinkan
    scraping_diizinkan = robot_parser.can_fetch(
        headers_scraping["User-Agent"],
        url_scraping
    )

    print("URL target:", url_scraping)
    print("Scraping diizinkan:", scraping_diizinkan)

    # Menyimpan bukti pemeriksaan robots
    robots_evidence = {
        "robots_url": robots_url,
        "target_url": url_scraping,
        "check_time": datetime.now(
            ZoneInfo("Asia/Makassar")
        ).isoformat(),
        "user_agent": headers_scraping["User-Agent"],
        "robots_status_code": response_robots.status_code,
        "allowed": scraping_diizinkan,
        "method": "urllib.robotparser"
    }

    os.makedirs("docs", exist_ok=True)

    with open(
        "docs/robots_check.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            robots_evidence,
            file,
            indent=4,
            ensure_ascii=False
        )

    print("\nBukti robots.txt tersimpan di docs/robots_check.json")

    if scraping_diizinkan:
        print("Target halaman diizinkan berdasarkan robots.txt.")
    else:
        print("Target halaman tidak diizinkan. Jangan lakukan scraping.")

else:
    print("robots.txt gagal diperiksa. Jangan lanjut scraping ulang.")

Status robots.txt: 200
URL target: https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population
Scraping diizinkan: True

Bukti robots.txt tersimpan di docs/robots_check.json
Target halaman diizinkan berdasarkan robots.txt.


In [20]:
# Sumber 3: Akuisisi halaman HTML untuk web scraping
import requests
import os
from datetime import datetime
from zoneinfo import ZoneInfo

# URL sumber web
url_scraping = (
    "https://en.wikipedia.org/wiki/"
    "List_of_Indonesian_cities_by_population"
)

# Identitas User-Agent menggunakan email yang sudah dimasukkan sebelumnya
headers_scraping = {
    "User-Agent": f"UTS-DAM/1.0 ({email})"
}

# Folder penyimpanan data mentah
os.makedirs("data/raw", exist_ok=True)

# Mengambil halaman HTML
response_html = requests.get(
    url_scraping,
    headers=headers_scraping,
    timeout=30
)

print("Status Code:", response_html.status_code)
print("Content-Type:", response_html.headers.get("Content-Type"))
print("Ukuran HTML:", len(response_html.content), "bytes")

# Memastikan halaman berhasil diakses
if response_html.status_code == 200:

    # Menyimpan HTML raw tanpa proses perubahan
    html_raw_path = "data/raw/city_population_raw.html"

    with open(html_raw_path, "wb") as file:
        file.write(response_html.content)

    print("\nAkuisisi HTML berhasil!")
    print("Raw HTML tersimpan:", html_raw_path)

else:
    print("\nAkuisisi gagal.")
    print(response_html.text[:500])

Status Code: 200
Content-Type: text/html; charset=UTF-8
Ukuran HTML: 287433 bytes

Akuisisi HTML berhasil!
Raw HTML tersimpan: data/raw/city_population_raw.html


In [21]:
# Ekstraksi dan pencocokan tabel populasi kota dari HTML
import pandas as pd
from io import StringIO

# Membaca tabel HTML
tables_html = pd.read_html(StringIO(response_html.text))

print("Jumlah tabel HTML ditemukan:", len(tables_html))

# Mengambil tabel utama populasi kota
table_index = 1
df_population_raw = tables_html[table_index].copy()

print("\nTabel yang dipilih:", table_index)
print("Jumlah baris tabel:", len(df_population_raw))
print("Kolom tabel:")
print(df_population_raw.columns.tolist())

# Menampilkan contoh data tabel
display(df_population_raw.head(5))

# Mengambil daftar kota dari dataset wisata
daftar_kota_dataset = df["City"].dropna().unique().tolist()

print("\nKota dalam dataset wisata:")
print(daftar_kota_dataset)

# Mencocokkan kota
df_population_match = df_population_raw[
    df_population_raw["City"].isin(daftar_kota_dataset)
].copy()

print("\nJumlah kota yang berhasil dicocokkan:",
      len(df_population_match))

print("\nHasil pencocokan kota:")
display(df_population_match)

Jumlah tabel HTML ditemukan: 9

Tabel yang dipilih: 1
Jumlah baris tabel: 95
Kolom tabel:
['City', 'Province', 'Region', '2020 census [2]', '2010 census [2]', 'Change 2010 to 2020', 'mid 2024 or 2025 official estimate', 'Notes']


,City,Province,Region,2020 census [2],2010 census [2],Change 2010 to 2020,mid 2024 or 2025 official estimate,Notes
0,Jakarta,Special Capital Region of Jakarta[3],Java,10562088,7007787,+50.72%,10684946,NaN
1,Surabaya,East Java,Java,2874314,2765487,+3.94%,2921996,NaN
2,Bekasi,West Java,Java,2543676,2334871,+8.94%,2644058,Satellite city of Jakarta
3,Bandung,West Java,Java,2444160,2394873,+2.06%,2528160,NaN
4,Medan,North Sumatra,Sumatra,2435252,2097610,+16.10%,2494512,Most populated city outside of Java



Kota dalam dataset wisata:
['Jakarta', 'Yogyakarta', 'Bandung', 'Semarang', 'Surabaya']

Jumlah kota yang berhasil dicocokkan: 5

Hasil pencocokan kota:


,City,Province,Region,2020 census [2],2010 census [2],Change 2010 to 2020,mid 2024 or 2025 official estimate,Notes
0,Jakarta,Special Capital Region of Jakarta[3],Java,10562088,7007787,+50.72%,10684946,NaN
1,Surabaya,East Java,Java,2874314,2765487,+3.94%,2921996,NaN
3,Bandung,West Java,Java,2444160,2394873,+2.06%,2528160,NaN
8,Semarang,Central Java,Java,1653524,1555984,+6.27%,1702379,NaN
34,Yogyakarta,Special Region of Yogyakarta,Java,373589,388627,−3.87%,375780,NaN


In [22]:
# Membersihkan dan menormalisasi hasil scraping populasi kota
# Mengambil kolom yang diperlukan saja
df_population = df_population_match[
    [
        "City",
        "Province",
        "2020 census [2]",
        "2010 census [2]",
        "Change 2010 to 2020",
        "mid 2024 or 2025 official estimate"
    ]
].copy()

# Mengganti nama kolom agar lebih sederhana
df_population.columns = [
    "City",
    "Province",
    "Population_2020",
    "Population_2010",
    "Population_Change",
    "Population_Estimate"
]

# Menghapus tanda koma jika data berbentuk teks
for kolom in ["Population_2020", "Population_2010", "Population_Estimate"]:
    df_population[kolom] = (
        df_population[kolom]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

# Mengubah kolom populasi menjadi numerik
for kolom in ["Population_2020", "Population_2010", "Population_Estimate"]:
    df_population[kolom] = pd.to_numeric(
        df_population[kolom],
        errors="coerce"
    )

print("Tabel hasil scraping setelah normalisasi:")
display(df_population)

print("\nInformasi tipe data:")
df_population.info()

Tabel hasil scraping setelah normalisasi:


,City,Province,Population_2020,Population_2010,Population_Change,Population_Estimate
0,Jakarta,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
1,Surabaya,East Java,2874314,2765487,+3.94%,2921996
3,Bandung,West Java,2444160,2394873,+2.06%,2528160
8,Semarang,Central Java,1653524,1555984,+6.27%,1702379
34,Yogyakarta,Special Region of Yogyakarta,373589,388627,−3.87%,375780



Informasi tipe data:
<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 34
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   City                 5 non-null      object
 1   Province             5 non-null      object
 2   Population_2020      5 non-null      int64 
 3   Population_2010      5 non-null      int64 
 4   Population_Change    5 non-null      object
 5   Population_Estimate  5 non-null      int64 
dtypes: int64(3), object(3)
memory usage: 452.0+ bytes


In [23]:
# Menyimpan dan memvalidasi hasil scraping populasi kota
import os

# Menyimpan hasil normalisasi
os.makedirs("data/processed", exist_ok=True)

population_csv_path = "data/processed/city_population_scraped.csv"

df_population.to_csv(
    population_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Hasil scraping berhasil disimpan!")
print("Lokasi:", population_csv_path)

# Validasi hasil scraping
print("\n=== VALIDASI DATA SCRAPING ===")

print("Jumlah baris:", len(df_population))
print("Jumlah kota unik:", df_population["City"].nunique())
print("Duplikasi baris:", df_population.duplicated().sum())

print("\nMissing value:")
display(df_population.isnull().sum())

print("\nDaftar kota:")
display(df_population["City"].tolist())

print("\nTabel hasil scraping:")
display(df_population)

Hasil scraping berhasil disimpan!
Lokasi: data/processed/city_population_scraped.csv

=== VALIDASI DATA SCRAPING ===
Jumlah baris: 5
Jumlah kota unik: 5
Duplikasi baris: 0

Missing value:


,0
City,0
Province,0
Population_2020,0
Population_2010,0
Population_Change,0
Population_Estimate,0



Daftar kota:


['Jakarta', 'Surabaya', 'Bandung', 'Semarang', 'Yogyakarta']


Tabel hasil scraping:


,City,Province,Population_2020,Population_2010,Population_Change,Population_Estimate
0,Jakarta,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
1,Surabaya,East Java,2874314,2765487,+3.94%,2921996
3,Bandung,West Java,2444160,2394873,+2.06%,2528160
8,Semarang,Central Java,1653524,1555984,+6.27%,1702379
34,Yogyakarta,Special Region of Yogyakarta,373589,388627,−3.87%,375780


In [27]:
# Menyimpan hasil scraping ke HTML berbentuk tabel

html_table_path = "data/processed/population_table.html"

df_population.to_html(
    html_table_path,
    index=False,
    border=1
)

print("HTML tabel berhasil disimpan:", html_table_path)

display(df_population)

HTML tabel berhasil disimpan: data/processed/population_table.html


,City,Province,Population_2020,Population_2010,Population_Change,Population_Estimate
0,Jakarta,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
1,Surabaya,East Java,2874314,2765487,+3.94%,2921996
3,Bandung,West Java,2444160,2394873,+2.06%,2528160
8,Semarang,Central Java,1653524,1555984,+6.27%,1702379
34,Yogyakarta,Special Region of Yogyakarta,373589,388627,−3.87%,375780


In [24]:
# Provenance Sumber 3: HTML Web Scraping
import os
import json
import hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

# Fungsi menghitung hash MD5
def hitung_md5_file(path):
    md5 = hashlib.md5()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            md5.update(chunk)

    return md5.hexdigest()

# Lokasi file
html_raw_path = "data/raw/city_population_raw.html"
population_csv_path = "data/processed/city_population_scraped.csv"

# Metadata sumber HTML
provenance_html = {
    "source_name": "List of Indonesian Cities by Population",
    "publisher": "Wikipedia contributors",
    "source_url": url_scraping,
    "acquisition_method": "HTTP GET Request menggunakan Python Requests",
    "extraction_method": "Pandas read_html",
    "table_index_extracted": 1,
    "total_html_tables_found": len(tables_html),
    "original_table_rows": len(tables_html[1]),
    "matched_cities": df_population["City"].tolist(),
    "matched_city_count": len(df_population),
    "data_fields": df_population.columns.tolist(),
    "access_time": datetime.now(
        ZoneInfo("Asia/Makassar")
    ).isoformat(),
    "collector_name": nama_pengambil,
    "collector_nim": nim_pengambil,
    "user_agent": f"UTS-DAM/1.0 ({email})",
    "raw_file": {
        "file_name": os.path.basename(html_raw_path),
        "file_size_bytes": os.path.getsize(html_raw_path),
        "md5": hitung_md5_file(html_raw_path)
    },
    "processed_file": {
        "file_name": os.path.basename(population_csv_path),
        "file_size_bytes": os.path.getsize(population_csv_path),
        "md5": hitung_md5_file(population_csv_path)
    },
    "processing_notes": (
        "Halaman HTML disimpan sebagai data raw. "
        "Tabel populasi kota diekstrak menggunakan Pandas read_html. "
        "Lima kota yang terdapat pada dataset destinasi wisata "
        "dicocokkan dan dipilih untuk proses integrasi selanjutnya."
    )
}

# Menyimpan provenance
os.makedirs("docs", exist_ok=True)

with open(
    "docs/provenance_html.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        provenance_html,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Provenance HTML berhasil dibuat!")
print(json.dumps(provenance_html, indent=4, ensure_ascii=False))

Provenance HTML berhasil dibuat!
{
    "source_name": "List of Indonesian Cities by Population",
    "publisher": "Wikipedia contributors",
    "source_url": "https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population",
    "acquisition_method": "HTTP GET Request menggunakan Python Requests",
    "extraction_method": "Pandas read_html",
    "table_index_extracted": 1,
    "total_html_tables_found": 9,
    "original_table_rows": 95,
    "matched_cities": [
        "Jakarta",
        "Surabaya",
        "Bandung",
        "Semarang",
        "Yogyakarta"
    ],
    "matched_city_count": 5,
    "data_fields": [
        "City",
        "Province",
        "Population_2020",
        "Population_2010",
        "Population_Change",
        "Population_Estimate"
    ],
    "access_time": "2026-11-26T19:15:48.226850+08:00",
    "collector_name": "Putu Reva Dhamma Nanda",
    "collector_nim": "2501010167",
    "user_agent": "UTS-DAM/1.0 (puturevadhammananda@gmail.com)",
    "raw_file"

In [25]:
# Integrasi tiga sumber data: Dataset utama, API cuaca, dan HTML scraping
import pandas as pd
import os
import json
import hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

# Pastikan pemeriksaan robots telah berhasil
assert scraping_diizinkan is True, (
    "Scraping tidak diizinkan oleh robots.txt. "
    "Proses integrasi dengan sumber scraping dihentikan."
)

# 1. Persiapan dan pemeriksaan kunci

# Menyamakan tipe data kunci Place_Id
df["Place_Id"] = pd.to_numeric(df["Place_Id"], errors="coerce")
df_weather["Place_Id"] = pd.to_numeric(
    df_weather["Place_Id"], errors="coerce"
)

# Memastikan kunci utama tidak duplikat
assert df["Place_Id"].is_unique, "Place_Id dataset utama duplikat"
assert df_weather["Place_Id"].is_unique, "Place_Id API duplikat"
assert df_population["City"].is_unique, "City scraping duplikat"

jumlah_awal = len(df)

# 2. Merge dataset utama dengan API

# Memilih kolom cuaca yang akan ditambahkan
kolom_cuaca = [
    "Place_Id",
    "Temperature_C",
    "Humidity_Percent",
    "Precipitation_mm",
    "Wind_Speed_kmh",
    "Weather_Time",
    "Timezone"
]

df_weather_merge = df_weather[kolom_cuaca].copy()

df_integrated = pd.merge(
    df,
    df_weather_merge,
    on="Place_Id",
    how="left",
    validate="one_to_one"
)

# Menghitung jumlah destinasi yang mendapat data cuaca
jumlah_cuaca = df_integrated["Temperature_C"].notna().sum()

print("Merge dataset utama + API berhasil")
print("Jumlah baris setelah merge API:", len(df_integrated))
print("Destinasi dengan data cuaca:", jumlah_cuaca)

# 3. Merge dengan hasil scraping populasi

df_integrated = pd.merge(
    df_integrated,
    df_population,
    on="City",
    how="left",
    validate="many_to_one"
)

# 4. Validasi hasil integrasi

print("\n=== VALIDASI INTEGRASI ===")
print("Jumlah baris dataset awal:", jumlah_awal)
print("Jumlah baris setelah integrasi:", len(df_integrated))
print("Jumlah Place_Id unik:", df_integrated["Place_Id"].nunique())
print("Jumlah duplikasi baris:", df_integrated.duplicated().sum())

print("\nDestinasi dengan data cuaca:",
      df_integrated["Temperature_C"].notna().sum())

print("Destinasi dengan data populasi:",
      df_integrated["Population_Estimate"].notna().sum())

# Memastikan integrasi tidak menambah/mengurangi baris dataset utama
assert len(df_integrated) == jumlah_awal, (
    "Jumlah baris berubah setelah merge!"
)

assert df_integrated["Place_Id"].nunique() == jumlah_awal, (
    "Ada Place_Id yang hilang atau terduplikasi!"
)

assert df_integrated.duplicated().sum() == 0, (
    "Ditemukan baris duplikat pada dataset terintegrasi!"
)

print("\nSemua validasi integrasi berhasil!")

# Menampilkan sampel hasil integrasi
display(df_integrated.head(10))

Merge dataset utama + API berhasil
Jumlah baris setelah merge API: 437
Destinasi dengan data cuaca: 20

=== VALIDASI INTEGRASI ===
Jumlah baris dataset awal: 437
Jumlah baris setelah integrasi: 437
Jumlah Place_Id unik: 437
Jumlah duplikasi baris: 0

Destinasi dengan data cuaca: 20
Destinasi dengan data populasi: 437

Semua validasi integrasi berhasil!


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,...,Humidity_Percent,Precipitation_mm,Wind_Speed_kmh,Weather_Time,Timezone,Province,Population_2020,Population_2010,Population_Change,Population_Estimate
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,NaN,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,60.0,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
5,6,Taman Impian Jaya Ancol,Taman Impian Jaya Ancol merupakan sebuah objek...,Taman Hiburan,Jakarta,25000,4.5,10.0,"{'lat': -6.117333200000001, 'lng': 106.8579951}",-6.117333,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
6,7,Kebun Binatang Ragunan,Kebun Binatang Ragunan adalah sebuah kebun bin...,Cagar Alam,Jakarta,4000,4.5,NaN,"{'lat': -6.3124593, 'lng': 106.8201865}",-6.312459,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
7,8,Ocean Ecopark,Ocean Ecopark Salah satu zona rekreasi Ancol y...,Taman Hiburan,Jakarta,180000,4.0,NaN,"{'lat': -6.125801699999999, 'lng': 106.8363249}",-6.125802,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
8,9,Pelabuhan Marina,Pelabuhan Marina Ancol berada di kawasan Taman...,Bahari,Jakarta,175000,4.4,NaN,"{'lat': 1.07888, 'lng': 103.931398}",1.078880,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946
9,10,Pulau Tidung,Pulau Tidung adalah salah satu kelurahan di ke...,Bahari,Jakarta,150000,4.5,NaN,"{'lat': -5.803205300000001, 'lng': 106.5237907}",-5.803205,...,NaN,NaN,NaN,NaN,NaN,Special Capital Region of Jakarta[3],10562088,7007787,+50.72%,10684946


In [26]:
# Menyimpan dataset terintegrasi dan metadata
import os
import json
import hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

os.makedirs("data/processed", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# Lokasi dataset final
integrated_path = "data/processed/tourism_integrated.csv"

# Menyimpan dataset hasil integrasi
df_integrated.to_csv(
    integrated_path,
    index=False,
    encoding="utf-8-sig"
)

# Fungsi menghitung MD5
def md5_integrasi(path):
    hash_md5 = hashlib.md5()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hash_md5.update(chunk)

    return hash_md5.hexdigest()

# Metadata integrasi
integration_metadata = {
    "project": "Akuisisi dan Integrasi Data Pariwisata Indonesia",
    "integration_time": datetime.now(
        ZoneInfo("Asia/Makassar")
    ).isoformat(),
    "sources": [
        {
            "source": "Indonesia Tourism Destination Dataset",
            "type": "CSV",
            "join_key": "Place_Id"
        },
        {
            "source": "Open-Meteo Weather API",
            "type": "API JSON",
            "join_key": "Place_Id",
            "coverage_records": int(jumlah_cuaca)
        },
        {
            "source": "Indonesian Cities Population",
            "type": "HTML scraping",
            "join_key": "City",
            "matched_cities": df_population["City"].tolist()
        }
    ],
    "merge_methods": [
        {
            "left_source": "Dataset utama",
            "right_source": "API cuaca",
            "key": "Place_Id",
            "how": "left",
            "validate": "one_to_one"
        },
        {
            "left_source": "Hasil merge dataset utama dan API",
            "right_source": "Scraping populasi",
            "key": "City",
            "how": "left",
            "validate": "many_to_one"
        }
    ],
    "row_count_before_merge": int(jumlah_awal),
    "row_count_after_merge": int(len(df_integrated)),
    "unique_place_id": int(df_integrated["Place_Id"].nunique()),
    "duplicate_rows": int(df_integrated.duplicated().sum()),
    "weather_records_matched": int(
        df_integrated["Temperature_C"].notna().sum()
    ),
    "population_records_matched": int(
        df_integrated["Population_Estimate"].notna().sum()
    ),
    "output_file": integrated_path,
    "output_file_size_bytes": os.path.getsize(integrated_path),
    "output_md5": md5_integrasi(integrated_path)
}

# Menyimpan metadata integrasi
with open(
    "docs/integration_metadata.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        integration_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Dataset final berhasil disimpan!")
print("Lokasi:", integrated_path)
print("Ukuran file:", os.path.getsize(integrated_path), "bytes")
print("MD5:", md5_integrasi(integrated_path))

print("\nMetadata integrasi tersimpan:")
print("docs/integration_metadata.json")

print("\nJumlah baris final:", len(df_integrated))
print("Jumlah kolom final:", len(df_integrated.columns))

Dataset final berhasil disimpan!
Lokasi: data/processed/tourism_integrated.csv
Ukuran file: 392345 bytes
MD5: bc12b9aa5d974befe85cc73f3248517d

Metadata integrasi tersimpan:
docs/integration_metadata.json

Jumlah baris final: 437
Jumlah kolom final: 24


In [29]:
from google.colab import files

readme_content = """# UTS Data Acquisition & Management
## Akuisisi dan Integrasi Data Pariwisata Indonesia

### Identitas
- **Nama:** Putu Reva Dhamma Nanda
- **NIM:** 2501010167
- **Mata Kuliah:** Data Acquisition & Management

## Deskripsi Proyek
Proyek ini membahas proses akuisisi dan integrasi data pariwisata Indonesia dari beberapa sumber. Data diperoleh melalui dataset CSV, API, dan web scraping. Data yang dikumpulkan mencakup informasi destinasi wisata, data cuaca, dan data populasi kota.

## Tujuan
1. Mengumpulkan data dari beberapa sumber berbeda.
2. Memahami karakteristik dan kualitas dataset.
3. Melakukan akuisisi data melalui API dan web scraping.
4. Melakukan pembersihan dan normalisasi data.
5. Mengintegrasikan data pariwisata, cuaca, dan populasi.
6. Menghasilkan dataset terintegrasi untuk analisis.

## Sumber Data

### Dataset Pariwisata
- Sumber: Kaggle
- URL: https://www.kaggle.com/datasets/aprabowo/indonesia-tourism-destination
- Metode: Download CSV

### Data Cuaca
- Sumber: Open-Meteo
- URL API: https://api.open-meteo.com/v1/forecast
- Dokumentasi: https://open-meteo.com/en/docs
- Metode: API Request menggunakan Requests
- Jumlah: 30 destinasi

### Data Populasi Kota
- Sumber: Wikipedia
- URL: https://en.wikipedia.org/wiki/List_of_Indonesian_cities_by_population
- Metode: Web Scraping menggunakan Requests dan Pandas

## Metode
1. Akuisisi dataset utama.
2. Pemeriksaan struktur dan kualitas data.
3. Dokumentasi provenance dan etika data.
4. Akuisisi data cuaca menggunakan API.
5. Pemeriksaan robots.txt.
6. Web scraping data populasi.
7. Pembersihan dan normalisasi.
8. Integrasi data cuaca berdasarkan Place_Id.
9. Integrasi data populasi berdasarkan City.
10. Validasi dan penyimpanan data akhir.

## Hasil
Dataset akhir merupakan data pariwisata yang telah diintegrasikan dengan data cuaca dan populasi kota.

File hasil integrasi:
`data/processed/tourism_integrated.csv`

Dokumentasi meliputi provenance, Data Ethics Checklist, robots.txt, dan metadata integrasi.

## Tools
- Python
- Google Colab
- Pandas
- NumPy
- Requests
- JSON
- Web Scraping
- Kaggle
- Open-Meteo API

## Catatan
Lisensi dataset utama perlu diverifikasi kembali pada halaman Kaggle. Proyek ini dibuat untuk keperluan pembelajaran dan tugas UTS Data Acquisition & Management.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md berhasil dibuat!")
files.download("README.md")

README.md berhasil dibuat!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_name = "UTS_DAM_Pariwisata"

drive_project = f"/content/drive/MyDrive/{project_name}"

os.makedirs(drive_project, exist_ok=True)

print("Folder proyek berhasil dibuat:")
print(drive_project)

Folder proyek berhasil dibuat:
/content/drive/MyDrive/UTS_DAM_Pariwisata


In [31]:
import os
import shutil

# Folder proyek di Google Drive
drive_project = "/content/drive/MyDrive/UTS_DAM_Pariwisata"

# File yang ingin diperbarui
files_to_copy = [
    "data/raw/open_meteo_raw.json",
    "data/processed/open_meteo_normalized.csv",
    "data/processed/weather_table.json",
    "data/processed/city_population_scraped.csv",
    "data/processed/population_table.html"
]

for file_path in files_to_copy:
    source = file_path
    destination = os.path.join(drive_project, file_path)

    os.makedirs(os.path.dirname(destination), exist_ok=True)

    shutil.copy2(source, destination)
    print("Berhasil diperbarui:", file_path)

Berhasil diperbarui: data/raw/open_meteo_raw.json
Berhasil diperbarui: data/processed/open_meteo_normalized.csv
Berhasil diperbarui: data/processed/weather_table.json
Berhasil diperbarui: data/processed/city_population_scraped.csv
Berhasil diperbarui: data/processed/population_table.html
